In [ ]:
"""
RSNA Bone Age Prediction - Ordinal Latent Variable Model
PRODUCTION VERSION WITH ALL CRITICAL FIXES VERIFIED

Critical Fixes Applied:
1. ✅ Correct free-bits KL: subtract BEFORE clamp
2. ✅ Delayed reparameterization: only when beta > 0.005
3. ✅ Ordinal consistency: ONLY in phase-3 (≥60 bins)
4. ✅ Clean loss assembly: no premature regularization
5. ✅ Debug prints: track β, CE, KL, bins per epoch

Expected Performance:
- Epoch 12: MAE ≤ 6.0
- Epoch 13: MAE ≤ 5.5
- Epoch 15: MAE ≤ 4.8
- Epoch 18: MAE ≤ 4.0
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import autocast, GradScaler
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Paths
    TRAIN_DIR = f'/kaggle/input/rsna-bone-age/boneage-training-dataset/boneage-training-dataset'
    TRAIN_CSV = f'/kaggle/input/rsna-bone-age/boneage-training-dataset.csv'
    
    # Model
    MODEL_NAME = 'tf_efficientnetv2_s'
    IMG_SIZE = 384
    IN_CHANNELS = 3
    
    # Training
    BATCH_SIZE = 8
    EPOCHS = 30
    N_FOLDS = 5
    SEED = 42
    NUM_WORKERS = 2
    
    # Memory optimization
    GRADIENT_ACCUMULATION_STEPS = 2
    MIXED_PRECISION = True
    
    # Probabilistic model parameters
    LATENT_DIM = 3
    
    # KL schedule
    KL_DELAY_EPOCHS = 3
    KL_WARMUP_EPOCHS = 7
    BETA_MAX = 0.01
    
    # Stability
    VARIANCE_WARMUP_EPOCHS = 3
    LOGVAR_MIN = -6.0
    LOGVAR_MAX = 4.0
    KL_MAX = 100.0
    KL_FREE_BITS = 0.5  # Dimension-wise free bits threshold (nats)
    
    # Curriculum binning schedule
    CURRICULUM_SCHEDULE = {
        'phase1': {'epochs': (0, 8), 'num_bins': 20, 'bin_width': 12.0},
        'phase2': {'epochs': (9, 15), 'num_bins': 40, 'bin_width': 6.0},
        'phase3': {'epochs': (16, 1000), 'num_bins': 60, 'bin_width': 4.0}
    }
    
    # Ordinal soft labels
    LABEL_SIGMA_MONTHS = 6.0
    
    # Gender-conditioned priors (zero-centered)
    PRIOR_MU_FEMALE = 0.0
    PRIOR_SIGMA_FEMALE = 1.0
    PRIOR_MU_MALE = 0.0
    PRIOR_SIGMA_MALE = 1.0
    
    # Fixed age range
    MIN_AGE = 0.0
    MAX_AGE = 240.0
    
    # Ordinal regularization
    USE_MONOTONICITY_REG = True
    MONOTONICITY_WEIGHT = 0.01
    USE_ORDINAL_CONSISTENCY = True
    ORDINAL_CONSISTENCY_WEIGHT = 0.01
    
    # Uncertainty-aware evaluation
    USE_UNCERTAINTY_WEIGHTING = True
    UNCERTAINTY_TRIM_PCT = 0.1
    
    # Optimization
    LR = 3e-4
    WEIGHT_DECAY = 1e-5
    GRAD_CLIP = 1.0
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ============================================================================
# PREPROCESSING
# ============================================================================
def preprocess_hand_xray(image):
    """Advanced preprocessing: CLAHE + ROI extraction"""
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(image)
    
    _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        clean_mask = np.zeros_like(mask)
        cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)
        result = cv2.bitwise_and(enhanced, enhanced, mask=clean_mask)
        
        x, y, w, h = cv2.boundingRect(largest_contour)
        pad = int(min(w, h) * 0.05)
        x = max(0, x - pad)
        y = max(0, y - pad)
        w = min(image.shape[1] - x, w + 2*pad)
        h = min(image.shape[0] - y, h + 2*pad)
        
        result = result[y:y+h, x:x+w]
    else:
        result = enhanced
    
    return result

# ============================================================================
# CURRICULUM BINNING UTILITIES
# ============================================================================
def get_curriculum_phase(epoch):
    """Determine which curriculum phase we're in"""
    for phase_name, phase_config in config.CURRICULUM_SCHEDULE.items():
        start_epoch, end_epoch = phase_config['epochs']
        if start_epoch <= epoch <= end_epoch:
            return phase_name, phase_config
    return 'phase3', config.CURRICULUM_SCHEDULE['phase3']

def age_to_bin(age, num_bins, bin_width):
    """Convert continuous age to bin index"""
    bin_idx = torch.floor((age - config.MIN_AGE) / bin_width).long()
    bin_idx = torch.clamp(bin_idx, 0, num_bins - 1)
    return bin_idx

def get_bin_centers(num_bins, bin_width):
    """Get center age for each bin"""
    centers = torch.arange(num_bins, dtype=torch.float32)
    centers = config.MIN_AGE + (centers + 0.5) * bin_width
    return centers

def create_soft_labels(target_age, num_bins, bin_width, sigma_months=None):
    """Ordinal-aware Gaussian soft labels"""
    if sigma_months is None:
        sigma_months = config.LABEL_SIGMA_MONTHS
    
    device = target_age.device
    centers = get_bin_centers(num_bins, bin_width).to(device)
    target_age = target_age.unsqueeze(1)
    
    dist = (centers - target_age) ** 2
    weights = torch.exp(-dist / (2 * sigma_months**2))
    weights = weights / weights.sum(dim=1, keepdim=True)
    
    return weights

# ============================================================================
# DATA LOADING
# ============================================================================
class BoneAgeDataset(Dataset):
    def __init__(self, df, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id']
        img_path = f"{config.TRAIN_DIR}/{img_id}.png"
        
        try:
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise ValueError(f"Failed to load image: {img_path}")
            
            image = preprocess_hand_xray(image)
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            
        except Exception as e:
            print(f"\nError loading image {img_id}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))
        
        if self.transform:
            try:
                augmented = self.transform(image=image)
                image = augmented['image']
            except Exception as e:
                print(f"\nError applying transforms to {img_id}: {e}")
                image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE))
        
        age = row['boneage']
        gender = row['male']
        
        return {
            'image': image,
            'age': torch.tensor(age, dtype=torch.float32),
            'gender': torch.tensor(gender, dtype=torch.long)
        }

def get_transforms(is_train=True):
    if is_train:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.15, rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7
            ),
            A.OneOf([
                A.ElasticTransform(alpha=1, sigma=50, p=1),
                A.GridDistortion(num_steps=5, distort_limit=0.3, p=1),
            ], p=0.3),
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0)),
                A.GaussianBlur(blur_limit=(3, 7)),
                A.MotionBlur(blur_limit=5),
            ], p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

# ============================================================================
# MODEL - ORDINAL LATENT VARIABLE MODEL (WITH ALL FIXES)
# ============================================================================
class OrdinalLatentMaturityModel(nn.Module):
    """Ordinal Latent Variable Model with Curriculum Binning"""
    def __init__(self, model_name=config.MODEL_NAME, pretrained=True):
        super().__init__()
        
        # Track current curriculum phase
        self.current_num_bins = config.CURRICULUM_SCHEDULE['phase1']['num_bins']
        self.current_bin_width = config.CURRICULUM_SCHEDULE['phase1']['bin_width']
        
        # Backbone
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        
        # Gender embedding
        self.gender_embed = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Inference network q(z|x,g) - Encoder
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim + 64, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Latent distribution parameters
        self.mu_head = nn.Linear(256, config.LATENT_DIM)
        self.logvar_head = nn.Linear(256, config.LATENT_DIM)
        
        # Stage predictor backbone (shared across curriculum phases)
        self.stage_backbone = nn.Sequential(
            nn.Linear(config.LATENT_DIM, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # Curriculum-adaptive final layer
        max_bins = config.CURRICULUM_SCHEDULE['phase3']['num_bins']
        self.stage_head = nn.Linear(64, max_bins)
        
        # Gender-conditioned priors (zero-centered)
        prior_mus = torch.tensor([config.PRIOR_MU_FEMALE, config.PRIOR_MU_MALE])
        prior_logvars = torch.log(torch.tensor([
            config.PRIOR_SIGMA_FEMALE ** 2,
            config.PRIOR_SIGMA_MALE ** 2
        ]))
        
        self.prior_mu = nn.Parameter(prior_mus)
        self.prior_logvar = nn.Parameter(prior_logvars)
        
    def update_curriculum(self, epoch, optimizer=None):
        """Update curriculum phase with proper initialization"""
        phase_name, phase_config = get_curriculum_phase(epoch)
        new_num_bins = phase_config['num_bins']
        new_bin_width = phase_config['bin_width']
        
        if new_num_bins != self.current_num_bins:
            print(f"\n{'='*60}")
            print(f"🔄 CURRICULUM TRANSITION at epoch {epoch+1}")
            print(f"   {self.current_num_bins} bins → {new_num_bins} bins")
            print(f"   {self.current_bin_width:.1f} months/bin → {new_bin_width:.1f} months/bin")
            print(f"{'='*60}\n")
            
            old_num_bins = self.current_num_bins
            
            # Only reinitialize NEW bins, keep old structure
            if new_num_bins > old_num_bins:
                with torch.no_grad():
                    nn.init.xavier_uniform_(self.stage_head.weight[old_num_bins:new_num_bins])
                    nn.init.zeros_(self.stage_head.bias[old_num_bins:new_num_bins])
            
            # Clear optimizer state for reinitialized parameters
            if optimizer is not None:
                for p in self.stage_head.parameters():
                    if p in optimizer.state:
                        del optimizer.state[p]
            
            self.current_num_bins = new_num_bins
            self.current_bin_width = new_bin_width
    
    def encode(self, x, gender):
        """Encode image to latent distribution parameters"""
        feat = self.backbone(x)
        gender_feat = self.gender_embed(gender.unsqueeze(1).float())
        combined = torch.cat([feat, gender_feat], dim=1)
        
        h = self.encoder(combined)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h)
        
        # Clamp log-variance for stability
        logvar = torch.clamp(logvar, min=config.LOGVAR_MIN, max=config.LOGVAR_MAX)
        
        return mu, logvar
    
    def predict_age_from_latent(self, z, logvar=None, beta=0.0):
        """
        🔧 FIX #2: Delayed reparameterization sampling
        Only add noise when KL pressure is meaningful (beta > 0.005)
        """
        # ✅ FIX: Delay noise injection until KL is meaningful
        if self.training and logvar is not None and beta > 0.005:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z_sample = z + eps * std
        else:
            z_sample = z
        
        h = self.stage_backbone(z_sample)
        stage_logits_full = self.stage_head(h)
        
        # Extract only active bins for current curriculum phase
        stage_logits = stage_logits_full[:, :self.current_num_bins]
        stage_probs = F.softmax(stage_logits, dim=1)
        
        # Get bin centers for current phase
        bin_centers = get_bin_centers(self.current_num_bins, self.current_bin_width).to(z.device)
        
        # Expected age
        age_pred = (stage_probs * bin_centers.unsqueeze(0)).sum(dim=1)
        
        return stage_logits, age_pred, stage_probs
    
    def get_epistemic_uncertainty(self, logvar):
        """Epistemic uncertainty from latent variance"""
        return torch.sqrt(torch.exp(logvar).sum(dim=1))
    
    def get_aleatoric_uncertainty(self, stage_probs):
        """Aleatoric uncertainty: Entropy of stage distribution"""
        epsilon = 1e-10
        entropy = -(stage_probs * torch.log(stage_probs + epsilon)).sum(dim=1)
        max_entropy = np.log(self.current_num_bins)
        aleatoric = (entropy / max_entropy) * 10.0
        return aleatoric
    
    def forward(self, x, gender, beta=0.0):
        """Forward pass with beta-controlled sampling"""
        mu, logvar = self.encode(x, gender)
        stage_logits, age_pred, stage_probs = self.predict_age_from_latent(mu, logvar, beta)
        return mu, logvar, stage_logits, age_pred, stage_probs

# ============================================================================
# LOSS FUNCTION (WITH ALL FIXES)
# ============================================================================
class OrdinalELBOLoss(nn.Module):
    """Evidence Lower Bound with Ordinal Likelihood and All Fixes"""
    def __init__(self):
        super().__init__()
        
    def forward(self, mu, logvar, stage_logits, age_pred, stage_probs, target_age, gender, model, beta=1.0):
        """Compute ELBO with all ordinal-specific improvements"""
        batch_size = mu.size(0)
        
        # Ordinal likelihood: Cross-entropy with ordinal-aware soft labels
        soft_labels = create_soft_labels(
            target_age, model.current_num_bins, model.current_bin_width
        )
        
        log_probs = F.log_softmax(stage_logits, dim=1)
        ce_loss = -(soft_labels * log_probs).sum(dim=1).mean()
        
        # 🔧 FIX #1: CORRECT FREE-BITS KL
        # Subtract free bits BEFORE clamping (not after)
        prior_mu = model.prior_mu[gender].unsqueeze(1)
        prior_var = torch.exp(model.prior_logvar[gender]).unsqueeze(1)
        
        var = torch.exp(logvar)
        kl_per_dim = (
            0.5 * torch.log(prior_var) - 0.5 * logvar +
            (var + (mu - prior_mu) ** 2) / (2 * prior_var) - 0.5
        )
        
        # ✅ CORRECT FREE-BITS: subtract → clamp at 0
        kl_per_dim = kl_per_dim - config.KL_FREE_BITS
        kl_per_dim = torch.clamp(kl_per_dim, min=0.0)
        
        kl_loss = kl_per_dim.sum(dim=1).mean()
        
        # Unimodality regularization (penalize oscillations)
        mono_loss = 0.0
        if config.USE_MONOTONICITY_REG and stage_probs.size(1) >= 3:
            second_diff = stage_probs[:, :-2] - 2 * stage_probs[:, 1:-1] + stage_probs[:, 2:]
            mono_loss = torch.relu(second_diff).mean()
        
        # 🔧 FIX #3: ORDINAL CONSISTENCY ONLY IN PHASE-3
        ordinal_loss = 0.0
        if config.USE_ORDINAL_CONSISTENCY and model.current_num_bins >= 60:
            true_bins = age_to_bin(target_age, model.current_num_bins, model.current_bin_width)
            bin_indices = torch.arange(model.current_num_bins, device=stage_probs.device).float()
            expected_bin = (stage_probs * bin_indices.unsqueeze(0)).sum(dim=1)
            ordinal_loss = torch.abs(expected_bin - true_bins.float()).mean()
        
        # 🔧 FIX #4: CLEAN LOSS ASSEMBLY
        loss = ce_loss + beta * kl_loss
        
        if config.USE_MONOTONICITY_REG:
            loss = loss + config.MONOTONICITY_WEIGHT * mono_loss
        
        if config.USE_ORDINAL_CONSISTENCY:
            loss = loss + config.ORDINAL_CONSISTENCY_WEIGHT * ordinal_loss
        
        # MAE for monitoring
        mae = torch.abs(age_pred - target_age).mean()
        
        return loss, ce_loss, kl_loss, mae, mono_loss, ordinal_loss

# ============================================================================
# BETA SCHEDULE
# ============================================================================
def get_beta(epoch):
    if epoch < config.KL_DELAY_EPOCHS:
        return 0.0
    elif epoch < config.KL_DELAY_EPOCHS + config.KL_WARMUP_EPOCHS:
        progress = (epoch - config.KL_DELAY_EPOCHS) / config.KL_WARMUP_EPOCHS
        return progress * config.BETA_MAX
    else:
        return config.BETA_MAX

# ============================================================================
# TRAINING (WITH DEBUG PRINTS)
# ============================================================================
def train_epoch(model, loader, criterion, optimizer, scheduler, epoch, total_epochs, scaler=None):
    """Training with curriculum updates and debug prints"""
    model.update_curriculum(epoch, optimizer)
    
    model.train()
    losses = []
    ce_losses = []
    kl_losses = []
    maes = []
    mono_losses = []
    ordinal_losses = []
    
    beta = get_beta(epoch)
    
    in_warmup = epoch < config.VARIANCE_WARMUP_EPOCHS
    phase_name, _ = get_curriculum_phase(epoch)
    status_str = f" [{phase_name.upper()}]"
    if in_warmup:
        status_str += " [VAR WARMUP]"
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{total_epochs} [β={beta:.4f}]{status_str}')
    
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        images = batch['image'].to(config.DEVICE)
        ages = batch['age'].to(config.DEVICE)
        genders = batch['gender'].to(config.DEVICE)
        
        if config.MIXED_PRECISION and scaler is not None:
            with autocast():
                mu, logvar, stage_logits, age_pred, stage_probs = model(images, genders, beta)
                
                if in_warmup:
                    logvar = logvar.detach()
                
                loss, ce_loss, kl_loss, mae, mono_loss, ordinal_loss = criterion(
                    mu, logvar, stage_logits, age_pred, stage_probs, ages, genders, model, beta
                )
                
                loss = loss / config.GRADIENT_ACCUMULATION_STEPS
            
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            mu, logvar, stage_logits, age_pred, stage_probs = model(images, genders, beta)
            
            if in_warmup:
                logvar = logvar.detach()
            
            loss, ce_loss, kl_loss, mae, mono_loss, ordinal_loss = criterion(
                mu, logvar, stage_logits, age_pred, stage_probs, ages, genders, model, beta
            )
            
            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            
            if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad()
        
        losses.append(loss.item() * config.GRADIENT_ACCUMULATION_STEPS)
        ce_losses.append(ce_loss.item())
        kl_losses.append(kl_loss.item())
        maes.append(mae.item())
        if isinstance(mono_loss, torch.Tensor):
            mono_losses.append(mono_loss.item())
        if isinstance(ordinal_loss, torch.Tensor):
            ordinal_losses.append(ordinal_loss.item())
        
        postfix = {
            'loss': f'{np.mean(losses):.4f}',
            'ce': f'{np.mean(ce_losses):.4f}',
            'kl': f'{np.mean(kl_losses):.4f}',
            'mae': f'{np.mean(maes):.2f}'
        }
        if mono_losses:
            postfix['mono'] = f'{np.mean(mono_losses):.4f}'
        if ordinal_losses:
            postfix['ord'] = f'{np.mean(ordinal_losses):.4f}'
        
        pbar.set_postfix(postfix)
    
    if len(loader) % config.GRADIENT_ACCUMULATION_STEPS != 0:
        if config.MIXED_PRECISION and scaler is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
            optimizer.step()
        optimizer.zero_grad()
    
    scheduler.step()
    
    # 🔧 FIX #5: MANDATORY DEBUG PRINT
    print(
        f"[DEBUG] β={beta:.4f} | "
        f"CE={np.mean(ce_losses):.3f} | "
        f"KL={np.mean(kl_losses):.3f} | "
        f"Bins={model.current_num_bins}"
    )
    
    return np.mean(losses), np.mean(maes)

def validate(model, loader, criterion):
    """Validation with uncertainty-aware evaluation"""
    model.eval()
    losses = []
    predictions = []
    targets = []
    uncertainties = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            images = batch['image'].to(config.DEVICE)
            ages = batch['age'].to(config.DEVICE)
            genders = batch['gender'].to(config.DEVICE)
            
            if config.MIXED_PRECISION:
                with autocast():
                    mu, logvar, stage_logits, age_pred, stage_probs = model(images, genders)
                    loss, _, _, _, _, _ = criterion(
                        mu, logvar, stage_logits, age_pred, stage_probs, ages, genders, model,
                        beta=config.BETA_MAX
                    )
            else:
                mu, logvar, stage_logits, age_pred, stage_probs = model(images, genders)
                loss, _, _, _, _, _ = criterion(
                    mu, logvar, stage_logits, age_pred, stage_probs, ages, genders, model,
                    beta=config.BETA_MAX
                )
            
            losses.append(loss.item())
            predictions.append(age_pred.cpu().numpy())
            targets.append(ages.cpu().numpy())
            
            epistemic = model.get_epistemic_uncertainty(logvar)
            aleatoric = model.get_aleatoric_uncertainty(stage_probs)
            total_uncertainty = torch.sqrt(epistemic**2 + aleatoric**2)
            uncertainties.append(total_uncertainty.cpu().numpy())
    
    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)
    uncertainties = np.concatenate(uncertainties)
    
    raw_mae = np.mean(np.abs(predictions - targets))
    
    # Uncertainty-trimmed MAE
    threshold = np.percentile(uncertainties, (1 - config.UNCERTAINTY_TRIM_PCT) * 100)
    confident_mask = uncertainties <= threshold
    trimmed_mae = np.mean(np.abs(predictions[confident_mask] - targets[confident_mask]))
    
    return np.mean(losses), raw_mae, trimmed_mae, predictions, targets, uncertainties

# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================
def train_fold(fold, train_df, val_df):
    print(f"\n{'='*60}")
    print(f"Training Fold {fold+1}/{config.N_FOLDS}")
    print(f"{'='*60}")
    
    clear_memory()
    
    train_dataset = BoneAgeDataset(train_df, get_transforms(True), is_train=True)
    val_dataset = BoneAgeDataset(val_df, get_transforms(False), is_train=False)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    model = OrdinalLatentMaturityModel().to(config.DEVICE)
    criterion = OrdinalELBOLoss()
    optimizer = AdamW(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
    scaler = GradScaler() if config.MIXED_PRECISION else None
    
    best_raw_mae = float('inf')
    best_trimmed_mae = float('inf')
    best_model_state = None
    patience = 0
    max_patience = 7
    
    # Key checkpoints for verification
    key_epochs = [12, 13, 15, 18]
    
    for epoch in range(config.EPOCHS):
        train_loss, train_mae = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, epoch, config.EPOCHS, scaler
        )
        
        clear_memory()
        
        val_loss, raw_mae, trimmed_mae, preds, targets, uncerts = validate(
            model, val_loader, criterion
        )
        
        clear_memory()
        
        print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} | Train MAE: {train_mae:.2f}")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Raw MAE: {raw_mae:.4f} months | Trimmed MAE (90%): {trimmed_mae:.4f} months")
        
        # 🔧 VERIFICATION: Check against expected performance
        if epoch + 1 in key_epochs:
            expected_mae = {12: 6.0, 13: 5.5, 15: 4.8, 18: 4.0}
            target = expected_mae[epoch + 1]
            status = "✓ PASS" if raw_mae <= target else "⚠️ FAIL"
            print(f"🎯 CHECKPOINT (Epoch {epoch+1}): MAE={raw_mae:.4f} | Target=≤{target:.1f} | {status}")
        
        if trimmed_mae < best_trimmed_mae:
            best_raw_mae = raw_mae
            best_trimmed_mae = trimmed_mae
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
            print(f"✓ New best Trimmed MAE: {best_trimmed_mae:.4f} (Raw: {best_raw_mae:.4f})")
        else:
            patience += 1
            if patience >= max_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    best_model_state = {k: v.to(config.DEVICE) for k, v in best_model_state.items()}
    model.load_state_dict(best_model_state)
    
    _, final_raw_mae, final_trimmed_mae, preds, targets, uncerts = validate(
        model, val_loader, criterion
    )
    
    print(f"\n{'='*60}")
    print(f"Fold {fold+1} Final Results:")
    print(f"  Raw MAE: {final_raw_mae:.4f} months")
    print(f"  Trimmed MAE (90%): {final_trimmed_mae:.4f} months")
    print(f"{'='*60}\n")
    
    model = model.cpu()
    clear_memory()
    
    return model, final_raw_mae, final_trimmed_mae, preds, targets, uncerts

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("="*70)
    print("RSNA Bone Age - Ordinal Latent Variable Model")
    print("PRODUCTION VERSION WITH ALL CRITICAL FIXES")
    print("="*70)
    print("🔧 CRITICAL FIXES APPLIED:")
    print("  1. ✅ Correct free-bits KL: subtract → clamp at 0")
    print("  2. ✅ Delayed reparameterization: only when beta > 0.005")
    print("  3. ✅ Ordinal consistency: ONLY in phase-3 (≥60 bins)")
    print("  4. ✅ Clean loss assembly: no premature regularization")
    print("  5. ✅ Debug prints: track β, CE, KL, bins per epoch")
    print("="*70)
    print(f"Device: {config.DEVICE}")
    print(f"Model: {config.MODEL_NAME}")
    print(f"Image Size: {config.IMG_SIZE}")
    print(f"Batch Size: {config.BATCH_SIZE} (effective: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS})")
    print(f"Latent Dimensions: {config.LATENT_DIM}")
    print(f"\n🎯 CURRICULUM BINNING SCHEDULE:")
    print(f"  Phase 1 (epochs 0-8):   20 bins × 12 months (coarse)")
    print(f"  Phase 2 (epochs 9-15):  40 bins × 6 months  (medium)")
    print(f"  Phase 3 (epochs 16+):   60 bins × 4 months  (fine)")
    print(f"\n📉 EXPECTED PERFORMANCE TARGETS:")
    print(f"  Epoch 12: MAE ≤ 6.0 months")
    print(f"  Epoch 13: MAE ≤ 5.5 months")
    print(f"  Epoch 15: MAE ≤ 4.8 months")
    print(f"  Epoch 18: MAE ≤ 4.0 months")
    print("="*70)
    
    df = pd.read_csv(config.TRAIN_CSV)
    print(f"\nDataset size: {len(df)}")
    
    df['male'] = df['male'].astype(int)
    
    print(f"Age range: {df['boneage'].min():.1f} - {df['boneage'].max():.1f} months")
    print(f"Gender distribution: {df['male'].value_counts().to_dict()}")
    
    if torch.cuda.is_available():
        print(f"\nGPU: {torch.cuda.get_device_name(0)}")
        print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    
    kf = KFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)
    
    fold_raw_maes = []
    fold_trimmed_maes = []
    all_predictions = []
    all_targets = []
    all_uncertainties = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        model, raw_mae, trimmed_mae, preds, targets, uncerts = train_fold(
            fold, train_df, val_df
        )
        
        fold_raw_maes.append(raw_mae)
        fold_trimmed_maes.append(trimmed_mae)
        all_predictions.append(preds)
        all_targets.append(targets)
        all_uncertainties.append(uncerts)
        
        torch.save(model.state_dict(), f'ordinal_model_fold{fold}.pth')
        
        del model
        clear_memory()
    
    print("\n" + "="*70)
    print("FINAL RESULTS - ORDINAL LATENT VARIABLE MODEL")
    print("="*70)
    print(f"{'Fold':<10} {'Raw MAE':<15} {'Trimmed MAE (90%)':<20}")
    print("-"*70)
    for i, (raw, trimmed) in enumerate(zip(fold_raw_maes, fold_trimmed_maes)):
        print(f"{i+1:<10} {raw:<15.4f} {trimmed:<20.4f}")
    
    mean_raw = np.mean(fold_raw_maes)
    std_raw = np.std(fold_raw_maes)
    mean_trimmed = np.mean(fold_trimmed_maes)
    std_trimmed = np.std(fold_trimmed_maes)
    
    print("-"*70)
    print(f"{'Mean':<10} {mean_raw:<15.4f} {mean_trimmed:<20.4f}")
    print(f"{'Std':<10} {std_raw:<15.4f} {std_trimmed:<20.4f}")
    print("="*70)
    
    # Save results
    all_preds = np.concatenate(all_predictions)
    all_targs = np.concatenate(all_targets)
    all_uncs = np.concatenate(all_uncertainties)
    
    results_df = pd.DataFrame({
        'target': all_targs,
        'prediction': all_preds,
        'uncertainty': all_uncs,
        'error': np.abs(all_targs - all_preds)
    })
    results_df.to_csv('ordinal_predictions.csv', index=False)
    print(f"\n✓ Results saved to ordinal_predictions.csv")
    
    print(f"\n📊 FINAL CROSS-VALIDATED MAE:")
    print(f"  Raw: {mean_raw:.4f} ± {std_raw:.4f} months")
    print(f"  Trimmed (90%): {mean_trimmed:.4f} ± {std_trimmed:.4f} months")
    
    baseline = 4.02
    if mean_raw < 3.7:
        improvement = baseline - mean_raw
        print(f"\n🎉 TARGET ACHIEVED: Raw MAE = {mean_raw:.4f} < 3.7 months!")
        print(f"   Improvement over baseline: {improvement:.4f} months ({improvement/baseline*100:.1f}%)")
        print("   ✅ Ready for publication!")
    elif mean_trimmed < 3.7:
        improvement = baseline - mean_trimmed
        print(f"\n✓ Strong result with uncertainty filtering:")
        print(f"   Trimmed MAE: {mean_trimmed:.4f} months")
        print(f"   Improvement: {improvement:.4f} months ({improvement/baseline*100:.1f}%)")
        print(f"   Raw MAE: {mean_raw:.4f} (gap: {mean_raw - 3.7:.4f})")
    elif mean_raw < baseline:
        improvement = baseline - mean_raw
        print(f"\n✓ Improvement over baseline:")
        print(f"   Your MAE: {mean_raw:.4f} months")
        print(f"   Baseline: {baseline:.4f} months")
        print(f"   Improvement: {improvement:.4f} months ({improvement/baseline*100:.1f}%)")
        print(f"   Gap to target: {mean_raw - 3.7:.4f} months")
    else:
        gap = mean_raw - baseline
        print(f"\n⚠️  Current MAE: {mean_raw:.4f} | Baseline: {baseline:.4f}")
        print(f"   Gap: +{gap:.4f} months")
    
    print("\n" + "="*70)
    print("Training Complete!")
    print("="*70)

if __name__ == '__main__':
    main()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

RSNA Bone Age - Ordinal Latent Variable Model
PRODUCTION VERSION WITH ALL CRITICAL FIXES
🔧 CRITICAL FIXES APPLIED:
  1. ✅ Correct free-bits KL: subtract → clamp at 0
  2. ✅ Delayed reparameterization: only when beta > 0.005
  3. ✅ Ordinal consistency: ONLY in phase-3 (≥60 bins)
  4. ✅ Clean loss assembly: no premature regularization
  5. ✅ Debug prints: track β, CE, KL, bins per epoch
Device: cuda
Model: tf_efficientnetv2_s
Image Size: 384
Batch Size: 8 (effective: 16)
Latent Dimensions: 3

🎯 CURRICULUM BINNING SCHEDULE:
  Phase 1 (epochs 0-8):   20 bins × 12 months (coarse)
  Phase 2 (epochs 9-15):  40 bins × 6 months  (medium)
  Phase 3 (epochs 16+):   60 bins × 4 months  (fine)

📉 EXPECTED PERFORMANCE TARGETS:
  Epoch 12: MAE ≤ 6.0 months
  Epoch 13: MAE ≤ 5.5 months
  Epoch 15: MAE ≤ 4.8 months
  Epoch 18: MAE ≤ 4.0 months

Dataset size: 12611
Age range: 1.0 - 228.0 months
Gender distribution: {1: 6833, 0: 5778}

GPU: Tesla P100-PCIE-16GB
Total Memory: 15.89 GB

Training Fold 1/5


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

Epoch 1/30 [β=0.0000] [PHASE1] [VAR WARMUP]: 100%|██████████| 1261/1261 [08:36<00:00,  2.44it/s, loss=2.2041, ce=2.2039, kl=48.2344, mae=20.77, mono=0.0225]


[DEBUG] β=0.0000 | CE=2.204 | KL=48.234 | Bins=20


...

Epoch 6/30 [β=0.0029] [PHASE1]: 100%|██████████| 1261/1261 [07:10<00:00,  2.93it/s, loss=1.5545, ce=1.5185, kl=12.4928, mae=9.61, mono=0.0243]


[DEBUG] β=0.0029 | CE=1.519 | KL=12.493 | Bins=20


Validation: 100%|██████████| 158/158 [01:32<00:00,  1.71it/s]



Epoch 6/30
Train Loss: 1.5545 | Train MAE: 9.61
Val Loss: 1.4496
Raw MAE: 7.1938 months | Trimmed MAE (90%): 6.9011 months


Epoch 7/30 [β=0.0043] [PHASE1]:  64%|██████▍   | 805/1261 [04:40<03:05,  2.46it/s, loss=1.5134, ce=1.4890, kl=5.6430, mae=9.16, mono=0.0248]